In [ ]:
df = pd.read_csv("/content/dataset.csv")

- **buisness_year**
송장이 생성된 연도
- **posting_date**
해당 송장이 ERP 데이터베이스에 입력된 날짜
전기일: 총계정원장에 기록되어 회계적으로 인식된 날짜
- **document_create_date**
송장 문서가 생성된 날짜
- **document_create_date.1**`document_create_date`를 정규화한 날짜 형식
- **due_in_date**
고객이 송장을 결제해야 하는 예정일
- **baseline_create_date**
송장이 생성된 기준 날짜
결제 기한 계산을 시작하는 기준점

In [ ]:
# 다중공선성 제거를 위해 중복 칼럼 삭제
columns_to_drop = ['document_create_date', 'document_create_date.1', 'posting_date']

In [ ]:
# baseline_create_date를 datetime 형식으로 변환
df['baseline_create_date'] = pd.to_datetime(df['baseline_create_date'], format='%Y%m%d', errors='coerce')

# 월, 일, 요일 추출
df['baseline_month'] = df['baseline_create_date'].dt.month
df['baseline_day'] = df['baseline_create_date'].dt.day
df['baseline_dayofweek'] = df['baseline_create_date'].dt.dayofweek

# print("새로운 피처 생성 완료: baseline_month, baseline_day, baseline_dayofweek")
# display(df[['baseline_create_date', 'baseline_month', 'baseline_day', 'baseline_dayofweek']].head())

In [ ]:
# due_in_date를 datetime 형식으로 변환
df['due_in_date'] = pd.to_datetime(df['due_in_date'], format='%Y%m%d', errors='coerce')

# Allowed_Pay_Days 계산 (일 단위)
df['Allowed_Pay_Days'] = (df['due_in_date'] - df['baseline_create_date']).dt.days

# print("새로운 피처 생성 완료: Allowed_Pay_Days")
# display(df[['baseline_create_date', 'due_in_date', 'Allowed_Pay_Days']].head())

In [ ]:
# 최종 확인(실행 안 해도 됨)

print("--- [날짜 컬럼 결측치 확인] ---")
display(df[['due_in_date', 'baseline_create_date']].isnull().sum())

print("\n--- [날짜 컬럼 범위 확인] ---")
# 최솟값과 최댓값을 확인하여 이상한 연도나 데이터가 있는지 봅니다.
date_summary = pd.DataFrame({
    'Min': [df['due_in_date'].min(), df['baseline_create_date'].min()],
    'Max': [df['due_in_date'].max(), df['baseline_create_date'].max()]
}, index=['due_in_date', 'baseline_create_date'])

display(date_summary)

# baseline_create_date가 due_in_date보다 늦은 데이터가 있는지 확인 (논리적 오류)
anomalous_dates = df[df['baseline_create_date'] > df['due_in_date']]
print(f"\n기준일이 마감일보다 늦은 데이터 건수: {len(anomalous_dates)}")

- document_create_date는 시스템 로그에 가까워서 별로 중요하지 않음
- 다중공선성 제거: [document_create_date, document_create_date.1, posting_date**] 삭제
	(모두 baseline_create_date와 상관계수 1)
- [due_in_date, baseline_create_date] 데이터 타입을 datetime으로 변경
- 계절성/패턴 추출을 위해 **baseline_create_date** 에서 월, 일, 요일 추출해 새 피쳐 생성 [baseline_month, baseline_day, baseline_dayofweek]
- **Allowed_Pay_Days** = due_in_date - baseline_create_date으로 새 칼럼 생성. 고객에게 주어진 총 결제 유예 기간
- dataset에는 넣지 않았으나 delay feature 생성해 Allowed_Pay_Days와 correlation matrix 그려봤을 때 -0.231
	- 고객에게 부여된 결제 유예 기간이 길수록 지연 일수가 줄어드는 경향 확인됨
- 추가 분석: delay & 특정 결제 조건(예: CAX2, NAX2 등)에서 지연이 두드러지게 발생하는 패턴을 확인

- <class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   business_code         50000 non-null  object        
 1   cust_number           50000 non-null  object        
 2   name_customer         50000 non-null  object        
 3   clear_date            40000 non-null  object        
 4   buisness_year         50000 non-null  float64       
 5   doc_id                50000 non-null  float64       
 6   due_in_date           50000 non-null  datetime64[ns]
 7   invoice_currency      50000 non-null  object        
 8   document type         50000 non-null  object        
 9   posting_id            50000 non-null  float64       
 10  area_business         0 non-null      float64       
 11  total_open_amount     50000 non-null  float64       
 12  baseline_create_date  50000 non-null  datetime64[ns]
 13  cust_payment_terms    50000 non-null  object        
 14  invoice_id            49994 non-null  float64       
 15  isOpen                50000 non-null  int64         
 16  baseline_month        50000 non-null  int32         
 17  baseline_day          50000 non-null  int32         
 18  baseline_dayofweek    50000 non-null  int32         
 19  Allowed_Pay_Days      50000 non-null  int64         
dtypes: datetime64[ns](2), float64(6), int32(3), int64(2), object(7)